<a href="https://colab.research.google.com/github/Prasad-Audipudi18/Faiss-with-hnsw-and-ivf/blob/main/storing_data_into_faiss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#installations
!pip install faiss-cpu
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.4 MB/s eta 0:00:00


In [ ]:
#required imports
import  os, time,psutil,numpy as np
import faiss
from sentence_transformers import SentenceTransformer
print(faiss.__version__)

1.15.1


In [ ]:
#Document
texts=[
    "Document about Ai simulations",
    "Deep learning and neural networks",
    "Graph-based indexing methods",
    "clustering algorithm in machinge learing"
]

In [ ]:
#Converts texts to numbers
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings=model.encode(texts)
embeddings= np.array(embeddings).astype("float32")
print(embeddings.shape, '2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(4, 384) 2


In [ ]:
#storing the vectors into faiss
dimension= embeddings.shape[1]
print(dimension)
index= faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("vectors stored:", index.ntotal)

384
vectors stored: 4


In [ ]:
query= "AI systems"
query_vector=model.encode([query]).astype("float32")
k=3
distances, indices=  index.search(query_vector,k)

In [ ]:
[texts[i] for i in  indices[0]]

['Document about Ai simulations',
 'Deep learning and neural networks',
 'clustering algorithm in machinge learing']

In [ ]:
#graph based search for faster HNSW
hsnw= faiss.IndexHNSWFlat(dimension, 32)
hsnw.add(embeddings)
distances,indices=hsnw.search(query_vector, k)

for i in indices[0]:
  print(texts[i])

Document about Ai simulations
Deep learning and neural networks
clustering algorithm in machinge learing


In [ ]:
#Distributed search IVF
nlist=2
quantizer=faiss.IndexFlatL2(dimension)
ivf= faiss.IndexIVFFlat(quantizer, dimension, nlist)
ivf.train(embeddings)
ivf.add(embeddings)
ivf.nprobe=1 # number of clusters to be searched
distances, indices=ivf.search(query_vector, k)
for i in indices[0]:
  print(texts[i])

Document about Ai simulations
Deep learning and neural networks
clustering algorithm in machinge learing
